In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re


In [2]:
df_final = pd.read_csv("general_info.csv")
df_filtered = (
    df_final
    .astype({"number_review":float})
    .loc[lambda df: df["number_review"]>100]
    .reset_index(drop=True)
    )
df_filtered

,company,website,rating,number_review,domain
0,Viator.com,www.viator.com,4.4,260639.0,['Travel Agency']
1,JustFly,justfly.com,4.2,194179.0,"['Travel Agency', 'Flights Search Site']"
2,Vegas.com,www.vegas.com,4.6,163448.0,['Travel Agency']
3,Gotogate,www.gotogate.com,3.1,141912.0,['Travel & Vacation']
4,Allianz Partners USA,www.allianztravelinsurance.com,4.1,126673.0,"['Travel Insurance Company', 'Financial Consul..."
...,...,...,...,...,...
1034,Buckingham Luxury Vacation Rentals,buckinghamtahoerentals.com,4.3,102.0,"['Vacation Rental', 'Travel Agency']"
1035,Descapada,descapada.com,4.2,102.0,"['Travel Aggregator', 'Tour Operator', 'Tour A..."
1036,Oru Kayak,www.orukayak.com,2.8,102.0,['Travel & Vacation']
1037,TRIPOMATE,www.tripomate.com,1.7,102.0,"['Travel Aggregator', 'Travel Agency']"


In [ ]:


# empty dict to prepare the df
df_prep = {
    "website": [],
    "5_star_percentage": [],
    "4_star_percentage": [],
    "3_star_percentage": [],
    "2_star_percentage": [],
    "1_star_percentage": [],    
}

url_base = "https://www.trustpilot.com/review/"

for index, website in enumerate(df_filtered["website"].to_list()):
    print(index, website)
    
    url_complete = url_base + website
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    # use sleep in case of ip bloackge (error 403)
    # time.sleep(random.uniform(5, 15))
    
    try:
        html = requests.get(url_complete, headers=headers, timeout=10)
        if html.status_code != 200:
            print(f"Error loading {html.status_code} at {index, website}")
            break  # oder break, je nach gewünschtem Verhalten

        soup = BeautifulSoup(html.text, "lxml")

        #extract relevant field that includes percentages
        field = soup.find("div", class_="paper_paper__EGeEb paper_outline__bqVmn card_card__yyGgu styles_reviewFilterCard__sn4Nz")
        
        if field:
            df_prep["website"].append(website)
            review_classes = re.findall(r'\d+%|<1%', field.text)
            if len(review_classes) == 5:
                df_prep["5_star_percentage"].append(review_classes[0])
                df_prep["4_star_percentage"].append(review_classes[1])
                df_prep["3_star_percentage"].append(review_classes[2])
                df_prep["2_star_percentage"].append(review_classes[3])
                df_prep["1_star_percentage"].append(review_classes[4])
            else:
                print(f"unexpected number of review classes at {website}: {review_classes}")
                continue
        else:
            print(f"field not found on {website}")
            break

    except Exception as e:
        print(f"error at {website}: {e}")
        continue

# DataFrame erzeugen
df_reviews = pd.DataFrame(df_prep)
display(df_reviews)
print(df_reviews.isna().sum())

0 www.flydealfare.com
Error loading 403 at (0, 'www.flydealfare.com')


,website,5_star_percentage,4_star_percentage,3_star_percentage,2_star_percentage,1_star_percentage


website              0.0
5_star_percentage    0.0
4_star_percentage    0.0
3_star_percentage    0.0
2_star_percentage    0.0
1_star_percentage    0.0
dtype: float64


In [ ]:
df_reviews = pd.DataFrame(df_prep)
#quick look
display(df_reviews)
print(df_reviews.isna().sum())

,website,5_star_percentage,4_star_percentage,3_star_percentage,2_star_percentage,1_star_percentage
0,www.flydealfare.com,95%,4%,<1%,<1%,<1%
1,peruforless.com,96%,3%,<1%,0%,0%
2,www.rockymountaingetaways.com,99%,<1%,<1%,0%,<1%
3,monarchairgroup.com,98%,2%,<1%,0%,<1%
4,incaexpert.com,98%,2%,0%,0%,0%
...,...,...,...,...,...,...
200,bigbearcoolcabins.com,63%,17%,10%,5%,5%
201,athotel.com,78%,10%,3%,2%,7%
202,furnishedquarters.com,62%,21%,8%,3%,6%
203,holidu.co.uk,65%,19%,6%,2%,8%


website              0
5_star_percentage    0
4_star_percentage    0
3_star_percentage    0
2_star_percentage    0
1_star_percentage    0
dtype: int64


In [ ]:
# df_reviews.to_csv("review_classes.csv",index=False)